In [ ]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

Starting virtual X frame buffer: Xvfb../xvfb: line 24: start-stop-daemon: command not found
.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [ ]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 8  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def ortho_init(module, gain=np.sqrt(2)):
    nn.init.orthogonal_(module.weight, gain=gain)
    if module.bias is not None:
        nn.init.constant_(module.bias, 0.0)

class NatureDQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            conv_out = self.conv(dummy)
            conv_out_size = conv_out.view(1, -1).size(1)
        self.fc = nn.Linear(conv_out_size, 512)
        self.policy = nn.Linear(512, n_actions)
        self.value = nn.Linear(512, 1)

        for layer in self.conv:
            if isinstance(layer, nn.Conv2d):
                ortho_init(layer)
        ortho_init(self.fc)
        ortho_init(self.policy, gain=0.01)
        ortho_init(self.value, gain=1.0)

    def forward(self, x):
        x = x / 255
        x = self.conv(x)
        x = F.relu(self.fc(x.view(x.size(0), -1)))
        logits = self.policy(x)
        value = self.value(x)
        return logits, value

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [ ]:
class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].

        if not isinstance(inputs, torch.Tensor):
            inputs = torch.from_numpy(inputs).float()
        else:
            inputs = inputs.float()
        with torch.no_grad():
            logits, values = self.model(inputs)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            actions = dist.sample()
            log_probs = dist.log_prob(actions)
        return {
            'actions': actions.cpu().numpy(),
            'logits': logits.cpu().numpy(),
            'log_probs': log_probs.cpu().numpy(),
            'values': values.cpu().numpy().squeeze(-1)
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [32]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [ ]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.

        obs_list = trajectory['observations']
        rewards = np.array(trajectory['rewards'])
        resets = np.array(trajectory['resets'])
        last_obs = trajectory['state']['latest_observation']

        T = len(obs_list)
        nenvs = obs_list[0].shape[0]
        last_obs_tensor = torch.from_numpy(last_obs).float()

        with torch.no_grad():
            last_values = self.policy.model(last_obs_tensor)[1].squeeze(-1)

        value_targets = np.zeros((T, nenvs), dtype=np.float32)
        next_values = last_values.cpu().numpy()
        for t in reversed(range(T)):
            if t + 1 < T:
                bootstrap = next_values * (1.0 - resets[t+1])
            else:
                bootstrap = next_values
            value_targets[t] = rewards[t] + self.gamma * bootstrap
            next_values = value_targets[t]
        trajectory['value_targets'] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [ ]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.

        for key, value in trajectory.items():
            if isinstance(value, list) and len(value) > 0:
                if isinstance(value[0], np.ndarray):
                    stacked = np.stack(value, axis=0)
                    merged = stacked.reshape(-1, *stacked.shape[2:])
                elif isinstance(value[0], torch.Tensor):
                    stacked = torch.stack(value, dim=0)
                    merged = stacked.view(-1, *stacked.shape[2:])
                else:
                    continue
                trajectory[key] = merged

        if 'value_targets' in trajectory and isinstance(trajectory['value_targets'], np.ndarray):
            vt = trajectory['value_targets']
            if vt.ndim == 2:
                trajectory['value_targets'] = vt.reshape(-1)
        return trajectory

In [ ]:
input_shape = (4, 84, 84)
n_actions = env.action_space(0).n
model = NatureDQN(input_shape, n_actions)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)

Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [ ]:
class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        log_probs = torch.from_numpy(trajectory['log_probs']).float()
        values = torch.from_numpy(trajectory['values']).float()
        value_targets = torch.from_numpy(trajectory['value_targets']).float()
        advantages = value_targets - values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        logits = torch.from_numpy(trajectory['logits']).float()
        probs = F.softmax(logits, dim=-1)
        log_probs_policy = F.log_softmax(logits, dim=-1)
        entropy = -(probs * log_probs_policy).sum(-1).mean()

        pg_loss = -(log_probs * advantages).mean()
        return pg_loss - self.entropy_coef * entropy

    def value_loss(self, trajectory):
        values = torch.from_numpy(trajectory['values']).float()
        value_targets = torch.from_numpy(trajectory['value_targets']).float()
        return F.mse_loss(values, value_targets)

    def loss(self, trajectory):
        return self.policy_loss(trajectory) + self.value_loss_coef * self.value_loss(trajectory)

    def step(self, trajectory):
        self.optimizer.zero_grad()
        total_loss = self.loss(trajectory)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        self.optimizer.step()
        return total_loss.item()

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [ ]:
#if you use TensorboardSummaries
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch.optim.lr_scheduler as lr_scheduler

total_timesteps = 10_000_000
nsteps = 5
nenvs = env.num_envs
n_updates = total_timesteps // (nsteps * nenvs)

optimizer = torch.optim.RMSprop(model.parameters(), lr=7e-4, alpha=0.99, eps=1e-5)
a2c = A2C(policy, optimizer)
scheduler = lr_scheduler.LambdaLR(optimizer, lambda update: 1.0 - update / n_updates)
writer = SummaryWriter(log_dir='logs')

runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=nsteps,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)

episode_rewards = []
for update in tqdm(range(1, n_updates + 1)):
    trajectory = runner.run()
    loss = a2c.step(trajectory)
    scheduler.step()
    global_step = update * nsteps * nenvs

    new_rewards = env.get_episode_rewards()
    if new_rewards:
        episode_rewards.extend(new_rewards)
        episode_rewards = episode_rewards[-100:]

    if update % 100 == 0:
        avg_reward = np.mean(episode_rewards) if episode_rewards else 0.0
        writer.add_scalar('avg_reward', avg_reward, global_step)
        writer.add_scalar('loss', loss, global_step)

writer.close()

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.